In [ ]:
import pandas as pd
import ast
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import scipy.stats as stats
from sklearn.preprocessing import PowerTransformer
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV
import statsmodels.api as sm
import math
import pickle

In [ ]:
data=pd.read_csv('parkinsons_disease_data_cls.csv')

In [ ]:
data['MedicalHistory'] = data['MedicalHistory'].apply(ast.literal_eval)
medical_history_data = pd.json_normalize(data['MedicalHistory'])
medical_history_data = medical_history_data.applymap(lambda x: 'Yes' if x == 'Yes' else 'No')
medical_history_data = medical_history_data.replace({'Yes': 1, 'No': 0})
data = pd.concat([data.drop(columns=['MedicalHistory']), medical_history_data], axis=1)

data['Symptoms'] = data['Symptoms'].apply(ast.literal_eval)
Symptoms_data = pd.json_normalize(data['Symptoms'])
Symptoms_data = Symptoms_data.applymap(lambda x: 'Yes' if x == 'Yes' else 'No')
Symptoms_data = Symptoms_data.replace({'Yes': 1, 'No': 0})
data = pd.concat([data.drop(columns=['Symptoms']), Symptoms_data], axis=1)

In [ ]:
def convert_to_minutes(time_str):
    hours, minutes = map(int, time_str.split(':'))
    return hours * 60 + minutes

data['WeeklyPhysicalActivity (hr)'] = data['WeeklyPhysicalActivity (hr)'].apply(convert_to_minutes)

In [ ]:
with open('preprocessing.pkl', 'rb') as f:
    obj = pickle.load(f)

In [ ]:
top_6_features = obj['top_6_features']
scaler = obj['scaler']
numerical_cols = obj['numerical_cols']
features_nulls = obj['features_nulls']
model = obj['model']

In [ ]:
top_6_features

,Feature,Importance
13,UPDRS,0.279377
15,FunctionalAssessment,0.104108
23,Tremor,0.103183
14,MoCA,0.094226
25,Bradykinesia,0.073066
24,Rigidity,0.071610


In [ ]:
for feature in top_6_features['Feature']:
  data[feature].fillna(features_nulls[feature], inplace=True)

In [ ]:
data[numerical_cols] = scaler.transform(data[numerical_cols])

In [ ]:
y = data['Diagnosis']

In [ ]:
print(top_6_features.columns)

Index(['UPDRS', 'FunctionalAssessment', 'Tremor', 'MoCA', 'Bradykinesia',
       'Rigidity'],
      dtype='object')


In [ ]:
data = data[top_6_features['Feature']]

In [ ]:
data

,UPDRS,FunctionalAssessment,Tremor,MoCA,Bradykinesia,Rigidity
0,0.020778,0.535827,0,0.955136,0,0
1,0.885720,0.993524,0,0.677473,0,0
2,0.669861,0.570783,1,0.687600,0,0
3,0.783830,0.725532,0,0.140790,0,1
4,0.248810,0.612302,0,0.716372,0,0
...,...,...,...,...,...,...
2020,0.282298,0.180601,1,0.923573,0,0
2021,0.699044,0.122151,0,0.511669,1,1
2022,0.261690,0.414107,1,0.925067,0,0
2023,0.318981,0.637421,1,0.800088,0,0


In [ ]:
data

,UPDRS,FunctionalAssessment,Tremor,MoCA,Bradykinesia,Rigidity
0,0.020778,0.535827,0,0.955136,0,0
1,0.885720,0.993524,0,0.677473,0,0
2,0.669861,0.570783,1,0.687600,0,0
3,0.783830,0.725532,0,0.140790,0,1
4,0.248810,0.612302,0,0.716372,0,0
...,...,...,...,...,...,...
2020,0.282298,0.180601,1,0.923573,0,0
2021,0.699044,0.122151,0,0.511669,1,1
2022,0.261690,0.414107,1,0.925067,0,0
2023,0.318981,0.637421,1,0.800088,0,0


In [ ]:
predections = model.predict(data)

In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y, predections))


0.9466666666666667
